$$\LARGE \textbf{CRISPR-VUS}$$

This notebook contains the computational framework underlying the CRISPR-VUS project. All results are available through the CRISPR-VUS portal: https://vus-portal.fht.org.

The project aims to assess the functional impact of variants of unknown significance in cancer by integrating CRISPR-based screening with genomic and pharmacological data. The resulting analyses are used to identify cancer-relevant variants, potential therapeutic targets, and opportunities for drug repositioning.


## 1. User settings and analysis parameters
These are the only settings that should be edited by the user before running the notebook.

In [1]:
options(warn = -1)  # Optional
library(yaml)
library(readr)

# Path to the configuration file
# Edit this path to point to the config.yaml file on your machine
# config_file <- "/path/to/your/crispr-vus/config.yaml"
config_file <- "/Users/letizia.depietri/OneDrive - Htechnopole/VUS PROJECT/crisprVUS_project/config.yaml"
config <- yaml::read_yaml(config_file)

# Set paths
home <- normalizePath(config$paths$home, mustWork = TRUE)
setwd(home)

pathdata <- file.path(home, config$paths$pathdata)
resultPath <- file.path(home, config$paths$resultPath)
raw_dir <- file.path(pathdata, config$paths$raw_dir)
robj_dir <- file.path(pathdata, config$paths$robj_dir)
figuresPath <- file.path(resultPath, config$paths$figuresPath)
tablesPath <- file.path(resultPath, config$paths$tablesPath)

# Main data directory containing raw and intermediate input files
if (!dir.exists(pathdata)) {
  stop("Data folder not found: ", pathdata)
}
if (!dir.exists(raw_dir)) {
  stop("Raw data folder not found: ", raw_dir)
}
if (!dir.exists(robj_dir)) {
  stop("Robj folder not found: ", robj_dir)
}

# Directory where analysis results will be written
if (!dir.exists(resultPath)) {
  dir.create(resultPath, recursive = TRUE)
}
if (!dir.exists(figuresPath)) {
  dir.create(figuresPath, recursive = TRUE)
}
if (!dir.exists(tablesPath)) {
  dir.create(tablesPath, recursive = TRUE)
}

# Analysis parameters - the default values are the ones that ensure reproducibility of published results
dep_threshold <- -0.5      # Maximum median DepMap fitness score across mutant CCLs (def = -0.5)
min_tissue_n <- 5          # Minimum number of samples required per tissue to be included in the analyses (def = 5)
download_latest <- FALSE   # Download new external source files or use local ones (def = FALSE)
RR_th <- 1.71              # Maximum rankRatio for DAM selection; lower values indicate stronger clustering of mutant CCLs among highly dependent cell lines (def = 1.71)
empirical_p_th <- 0.2      # Maximum empirical permutation p-value for DAM selection (def = 0.2)
hypergeom_p_th <- 0.2      # Maximum hypergeometric tail p-value for DAM selection (def = 0.2)
n_rantrials <- 1000        # Number of permutations for DAMs empirical p-value estimation (def = 1000)
produce_plots <- TRUE      # Whether to generate any plot during execution (def = TRUE). Plots are saved in figuresPath.
sel_tissue_idx <- NULL     # NULL = analyse all eligible tissues; otherwise provide indices from the tissue list below, e.g. c(6, 24, 28) (def = NULL)

Available tissues.

Select tissues using their indices in `sel_tissue_idx`.

| | |
|---|---|
| 1. Acute Myeloid Leukemia | 19. Hepatocellular Carcinoma |
| 2. B-Cell Non-Hodgkin's Lymphoma | 20. Kidney Carcinoma |
| 3. B-Lymphoblastic Leukemia | 21. Melanoma |
| 4. Biliary Tract Carcinoma | 22. Mesothelioma |
| 5. Bladder Carcinoma | 23. Neuroblastoma |
| 6. Breast Carcinoma | 24. Non-Small Cell Lung Carcinoma |
| 7. Burkitt's Lymphoma | 25. Oral Cavity Carcinoma |
| 8. Cervical Carcinoma | 26. Osteosarcoma |
| 9. Chronic Myelogenous Leukemia | 27. Ovarian Carcinoma |
| 10. Colorectal Carcinoma | 28. Pancreatic Carcinoma |
| 11. Endometrial Carcinoma | 29. Plasma Cell Myeloma |
| 12. Esophageal Carcinoma | 30. Prostate Carcinoma |
| 13. Esophageal Squamous Cell Carcinoma | 31. Rhabdomyosarcoma |
| 14. Ewing's Sarcoma | 32. Small Cell Lung Carcinoma |
| 15. Gastric Carcinoma | 33. Squamous Cell Lung Carcinoma |
| 16. Glioblastoma | 34. T-Cell Non-Hodgkin's Lymphoma |
| 17. Glioma | 35. T-Lymphoblastic Leukemia |
| 18. Head and Neck Carcinoma | 36. Thyroid Gland Carcinoma |

## 2. Data loading
If download_latest = TRUE, the most recent data files are downloaded into the raw data directory. Otherwise, previously downloaded local files are used.

In [2]:
source("./R/2_data_loading.R")

if (download_latest) {
    # Download latest files
    data_paths <- download_latest_inputs(raw_dir = raw_dir)

    gene_annot_file <- data_paths$gene_annot_path
    CMP_annot_file <- data_paths$model_list_path
    crispr_file <- data_paths$crispr_path
    cl_variants_file <- data_paths$cl_variants_path
    gdsc1_file <- data_paths$gdsc1_path
    gdsc2_file <- data_paths$gdsc2_path
    intogen_file <- data_paths$intogen_path
} else {
    # Use local files
    gene_annot_file <- file.path(raw_dir, config$files$gene_annot_file)
    CMP_annot_file <- file.path(raw_dir, config$files$CMP_annot_file)
    crispr_file <- file.path(raw_dir, config$files$crispr_file)
    cl_variants_file <- file.path(raw_dir, config$files$cl_variants_file)
    gdsc1_file <- file.path(raw_dir, config$files$gdsc1_file)
    gdsc2_file <- file.path(raw_dir, config$files$gdsc2_file)
    intogen_file <- file.path(raw_dir, config$files$intogen_file)
}

# - Load datasets -

# Gene annotation
gene_annot <- read_csv(gene_annot_file, show_col_types = FALSE)

# Cell model annotation
CMP_annot0 <- read_csv(CMP_annot_file, show_col_types = FALSE)

# CRISPR dependency scores
scaled_depFC <- read.csv(crispr_file, row.names = 1)
colnames(scaled_depFC) <- gsub("\\..*", "", colnames(scaled_depFC))

# Cell line variants
cl_variants0 <- read_csv(cl_variants_file, show_col_types = FALSE)
cl_variants <- cbind(cl_variants0, gene_annot$hgnc_symbol[match(cl_variants0$gene_id, gene_annot$gene_id)])
colnames(cl_variants)[ncol(cl_variants)] <- 'gene_symbol_2023'

# GDSC dose-response
gdsc1 <- read.csv(gdsc1_file, header=TRUE, stringsAsFactors=FALSE)
gdsc2 <- read.csv(gdsc2_file, header=TRUE, stringsAsFactors=FALSE)

# IntOGen driver genes
intogen_drivers <- read.table(intogen_file, header=TRUE, sep="\t", stringsAsFactors=FALSE)
intogen_drivers_s <- unique(sort(intogen_drivers$SYMBOL))

# Load precomputed objects
load(file.path(robj_dir, config$robj_files$ADaM))
load(file.path(robj_dir, config$robj_files$FiPer_outputs))
load(file.path(robj_dir, config$robj_files$basal_exp))
load(file.path(pathdata, config$robj_files$ensembl))

## 3. Preprocessing
Preprocessing of CRISPR dependency matrix and selection of cell lines with complete information.

In [3]:
source("./R/3_preprocessing.R")

# Preprocess the CRISPR dependency matrix
crispr_processed <- preprocess_crispr_matrix(scaled_depFC = scaled_depFC, CMP_annot0 = CMP_annot0, cl_variants = cl_variants, threshold = dep_threshold)

scaled_depFC <- crispr_processed$scaled_depFC   # Clean/scaled dependency matrix
bdep <- crispr_processed$bdep                   # Binarized dependency matrix
CMP_annot <- crispr_processed$CMP_annot         # Filtered CMP annotation aligned with CRISPR data

2161 annotated models in the Cell Models Passports
of which 1430 with mutation data
of which 1078 with high quality CRISPR data


## 4. Tissue selection
This excludes generic, uninformative, or underrepresented cancer categories. If sel_tissue_idx is provided, at this stage such subselection also happens.

In [4]:
source("./R/4_tissue_selection.R")

# Select cancer tissues for downstream analysis
tissue_selection <- select_and_save_tissues(CMP_annot = CMP_annot, min_n = min_tissue_n, sel_tissue_idx = sel_tissue_idx)

# Update objects with selected tissues
CMP_annot <- tissue_selection$CMP_annot
tissues <- tissue_selection$tissues

tissues <- sort(tissues)
ntiss <- length(tissues)

# Saving
incl_cl_annot <- CMP_annot
save(incl_cl_annot, file = file.path(resultPath, "_incl_cl_annot.RData"))
write.table(incl_cl_annot, sep = "\t", quote = FALSE, row.names = TRUE, col.names = NA, file = file.path(resultPath, "_incl_cl_annot.tsv"))

## 5. DAM tissue-specific analysis
Retrieving tissue-specific CRISPR dependencies for each gene.


In [ ]:
source("./R/5_DAM_analysis.R")

# Create CRISPR plots folder
CRISPRplotsPath <- file.path(resultPath, config$paths$CRISPRplots_dir)
if (!dir.exists(CRISPRplotsPath)) {
  dir.create(CRISPRplotsPath, recursive = TRUE)
}

# Necessary to ensure reproducibility of empirical p values / FDR values
# The original seed is used only for a fresh run.
# If resuming, the exact RNG state saved after the last completed tissue is restored.
rng_file <- file.path(resultPath, "_CRISPR_rng_state_after_last_completed_tissue.RData")
progress_file <- file.path(resultPath, "_CRISPR_completed_tissues.txt")

if (file.exists(rng_file)) {
  load(rng_file)
} else {
  set.seed(123)
}

completed <- character(0)
if (file.exists(progress_file)) {
  completed <- readLines(progress_file)
}

for (t_idx in seq_along(tissues)) {
  # Loops over genes in a tissue to find tissue-specific CRISPR dependencies
  # For each gene:
  # -extracts mutations in that gene (variantSpectrum function)
  # -checks which cell lines are positive for mutations (positives function)
  # -tests whether mutations induce dependency in CRISPR data (Optimal_clasTest / clasTest functions)
  # -computes statistics
  # -outputs RESTOT, a tissue-specific table with DAMs and their scores

  ctiss <- tissues[t_idx]
  dir.create(file.path(CRISPRplotsPath,ctiss))

  if (ctiss %in% completed) {
    cat("Skipping completed tissue:", ctiss, "\n")
    next
  }

  crispr_res <- process_tissue(ctiss = ctiss, scaled_depFC = scaled_depFC, bdep = bdep, CMP_annot = CMP_annot, cl_variants = cl_variants, CRISPRplotsPath = CRISPRplotsPath, RR_th = RR_th,
    dep_threshold = dep_threshold, n_rantrials = n_rantrials, produce_plots = produce_plots, ADaM = ADaM, Perc_AUC = Perc_AUC, t_idx = t_idx, ntiss = ntiss)

  # Save tissue-specific CRISPR results
  RESTOT <- crispr_res$RESTOT
  ts_cl_variants <- crispr_res$ts_cl_variants
  DAM_bearing_genes <- crispr_res$DAM_bearing_genes

  save(RESTOT, file = file.path(resultPath, paste0(ctiss, '_results.RData')))
  save(ts_cl_variants, file = file.path(resultPath, paste0(ctiss, '_testedVariants.RData')))
  write.table(RESTOT, quote = FALSE, sep = '\t', row.names = FALSE, file = file.path(resultPath, paste0(ctiss, '_results.tsv')))
  write.table(ts_cl_variants, quote = FALSE, sep = '\t', row.names = FALSE, file = file.path(resultPath, paste0(ctiss, '_testedVariants.tsv')))
  save(DAM_bearing_genes, file = file.path(resultPath, paste0(ctiss, '_DAM_bearing_genes.RData')))

  # Ensures reproducibility
  completed <- c(completed, ctiss)
  writeLines(completed, progress_file)

  save(.Random.seed, file = rng_file)

}

## 6. Integration with IntOGen driver information
Annotating each gene with IntOGen driver roles and writing to tissue result tables.

In [ ]:
source("./R/6_IntOGen_drivers.R")

for(ctiss in tissues) {
  # Iterates over all genes in the RESTOT table for a given tissue
  # For each gene:
  # -checks if it is listed in IntOGen cancer drivers
  # -extracts tumor types in which the gene is a driver and its role breakdown (activating, loss-of-function, ambiguous)
  # -adds IntOGen annotations to RESTOT

  # Load RESTOT for this tissue
  load(file.path(resultPath, paste0(ctiss, "_results.RData")))

  RESTOT <- annotate_IntOGen(RESTOT = RESTOT, intOGen_drivers = intogen_drivers)

  # Saving updated RESTOT
  save(RESTOT, file = file.path(resultPath, paste0(ctiss, '_results.RData')))
  write.table(RESTOT, quote=FALSE, sep='\t', row.names = FALSE, file = file.path(resultPath, paste0(ctiss,'_results.tsv')))

}

## 7. Drug response analysis
Identifying SAMs for each tissue.


In [ ]:
set.seed(123)
source("./R/7_SAM_analysis.R")

# Load drug-target info
drugTargetInfo <- read.table(file.path(raw_dir, config$files$drug_target_info_file), sep = "\t", header = TRUE, stringsAsFactors = FALSE)

# Prepare GDSC combined dataset
gdsc1$uDRUG_ID <- paste0(gdsc1$DRUG_ID, "_gdsc1")
gdsc2$uDRUG_ID <- paste0(gdsc2$DRUG_ID, "_gdsc2")
gdscAll <- rbind(gdsc1, gdsc2)

# Filter drugTargetInfo for drugs present in GDSC
drugTargetInfo <- drugTargetInfo[drugTargetInfo$Drug.ID %in% union(gdsc1$DRUG_ID, gdsc2$DRUG_ID), ]

# Create DR plots folder
DRplotsPath <- file.path(resultPath, config$paths$DRplots_dir)
if (!dir.exists(DRplotsPath)) {
  dir.create(DRplotsPath, recursive = TRUE)
}

for (t_idx in seq_along(tissues)) {

  # Loops over tissues to identify SAMs
  # For each tissue:
  # -loads previously identified DAMs (RESTOT)
  # -selects strongest mutation-dependent genes
  # -retrieves drugs targeting those genes
  # -tests whether mutant cell lines are more drug-sensitive
  # -computes rank-based, hypergeometric and empirical statistics
  # -saves detailed drug screening results

  ctiss <- tissues[t_idx]

  load(file.path(resultPath, paste0(ctiss, "_results.RData")))

  dir.create(file.path(DRplotsPath,ctiss))

  RES_DR <- analyze_drugs(RESTOT = RESTOT, CMP_annot = CMP_annot, drugTargetInfo = drugTargetInfo, gdscAll = gdscAll, DRplotsPath = DRplotsPath,
    ctiss = ctiss, RR_th = RR_th, dep_threshold = dep_threshold, hypTest_p = hypergeom_p_th, empPval = empirical_p_th, drug_RR = 1.5, drug_HG_pval = 0.2,
    drug_EMP_pval = 0.2, n_rantrials = n_rantrials, display = produce_plots, tissue_idx = t_idx, ntiss = ntiss)

  for (i in seq_along(RES_DR)) {
    SCREENdata <- RES_DR[[i]]$SCREENdata
    save_file <- RES_DR[[i]]$save_file
    save(SCREENdata, file = file.path(DRplotsPath, ctiss, save_file))
  }

}


## 8. Post-processing, first summaries, and driver enrichment analyses
Summarizing how many variants, hits, DAMs, DAM-bearing genes, cell lines, and cancer types were included. Reporting also global and tissue-specific enrichment analyses for IntOGen drivers.

In [5]:
source("./R/8_postprocessing.R")
set.seed(123)
load(file.path(resultPath, '_incl_cl_annot.RData'))

# Cell line summary
ss <- summarize_cell_lines(incl_cl_annot = incl_cl_annot, figuresPath = figuresPath, produce_plots = produce_plots)

# Collect and summarize tested variants
totalTestedVariants <- collect_tested_variants(resultPath = resultPath)
save(totalTestedVariants, file = file.path(resultPath, "_totalTestedVariants.RData"))
write.table(totalTestedVariants, sep = "\t", quote = FALSE, file = file.path(resultPath, "_totalTestedVariants.tsv"))

res_tested <- summarize_tested_variants(totalTestedVariants = totalTestedVariants, figuresPath = figuresPath, tissues = tissues, produce_plots = produce_plots)
totalTestedVariants <- res_tested$totalTestedVariants
ct_tested_variants <- res_tested$ct_tested_variants
ntestedGenes <- res_tested$ntestedGenes

# Collect all DAMs / hits / DAM bearing genes
res_dams <- collect_all_DAMs(resultPath = resultPath, totalTestedVariants = totalTestedVariants, RR_th = RR_th, dep_threshold = dep_threshold, empPval = empirical_p_th, hypTest_p = hypergeom_p_th)
allHits <- res_dams$allHits
allDAMs <- res_dams$allDAMs
allDAM_bearing_genes <- res_dams$allDAM_bearing_genes

save(allHits, file = file.path(resultPath, "_allHits.RData"))
write.table(allHits, sep = "\t", quote = FALSE, file = file.path(resultPath, "_allHits.tsv"))
save(allDAMs, file = file.path(resultPath, "_allDAMs.RData"))
write.table(allDAMs, sep = "\t", quote = FALSE, file = file.path(resultPath, "_allDAMs.tsv"))
save(allDAM_bearing_genes, file = file.path(resultPath, "_allDAM_bearing_genes.RData"))
write.table(allDAM_bearing_genes, quote = FALSE, sep = "\t", file = file.path(resultPath, "_allDAM_bearing_genes.tsv"))

# Global DAM summaries
summarize_DAMs(allHits = allHits, allDAMs = allDAMs, totalTestedVariants = totalTestedVariants, ntestedGenes = ntestedGenes)

# Plots by cancer type
clc <- read.xlsx(file.path(raw_dir, config$files$cl_tissue_colors_file), sheet = 2, rowNames = TRUE)
plot_DAMs_by_ctype(allDAMs = allDAMs, clc = clc, figuresPath = figuresPath, produce_plots = produce_plots)

# DAM counts per tissue
res_counts <- compute_DAM_stats(allDAMs = allDAMs, tissues = tissues)
nDAMbearing <- res_counts$nDAMbearing
nDAMs <- res_counts$nDAMs

# Global enrichment of IntOGen cancer drivers among DAM-bearing genes
enrich_global <- intOGen_global_enrichment(intogen_drivers = intogen_drivers_s, allDAM_bearing_genes = allDAM_bearing_genes, ntestedGenes = ntestedGenes)
DAMbearing_known_as_cancer_driver <- enrich_global$x
k <- enrich_global$k
N <- enrich_global$N

# IntOGen role-specific enrichment
enrich_roles <- intOGen_role_enrichment(intogen_drivers = intogen_drivers, allDAM_bearing_genes = allDAM_bearing_genes, k = k, N = N,
                                        x_global = DAMbearing_known_as_cancer_driver, figuresPath = figuresPath, produce_plots = produce_plots)

# Tissue-specific IntOGen enrichment analysis
cancer_match <- read.csv(file.path(raw_dir, config$files$cancer_match_file_v3), header = TRUE, row.names = 1, sep = ";")
res_comp <- composition_analysis(intogen_drivers = intogen_drivers, ct_mapping = cancer_match, figuresPath = figuresPath, tissues = tissues, allHits = allHits,
                                 totalTestedVariants = totalTestedVariants, produce_plots = produce_plots)
COMPOSITION <- res_comp$COMPOSITION
COMPOSITIONp <- res_comp$COMPOSITIONp
CT_DAM_Bearing <- res_comp$CT_DAM_Bearing

977 cell lines included in the analysis
36 considered cancer types
median number of cell lines per cancer type = 24 
min = 6 for Burkitt's Lymphoma 
max = 69 for Non-Small Cell Lung Carcinoma 
532331 tested variants x cancer type combos
403269 tested individual variants
involving 16842 genes
1899 hits with rankratio and medFitness effect below the thresholds (default: 1.71 and -0.5 respectively), empP < 0.20 and HG P < 0.20 across cancer types
2376 individual DAMs across cancer types
corresponding to 2284 individual variants
0.4463388 % of tested cases (considering variants tested multiple times across different cancer types)
involving 1383 genes
8.211614 % of tested cases
49.5 median number of DAM bearing genes across cancer types
min = 7 for T-Cell Non-Hodgkin's Lymphoma 
max = 167 for Colorectal Carcinoma 
57.5 median number of DAMs across cancer types
min = 7 for T-Cell Non-Hodgkin's Lymphoma 
max = 245 for Colorectal Carcinoma 
Of the 1383 DAM bearing genes, 123 have been previous

## 9. Global summaries and visualization
Summarizing global DAM/DAM bearing gene recurrence, expression and essentiality patterns, hit rankings, SNR across cancer types, and genomic-noise group comparisons.

In [ ]:
source("./R/9_visualization.R")
load(file.path(resultPath, '_allHits.RData'))
load(file.path(resultPath, '_allDAMs.RData'))
load(file.path(resultPath, '_totalTestedVariants.RData'))

clc <- read.xlsx(file.path(raw_dir, config$files$cl_tissue_colors_file), sheet = 2, rowNames = TRUE)

# Recurrence analyses for DAMs and DAM bearing genes
dam_occurrencies <- plot_nDAMs_in_nCtypes(allDAMs = allDAMs, figuresPath = figuresPath, produce_plots = produce_plots)
dam_bg <- compute_DAMbgs_across_analyses(allHits = allHits)
plot_nDAMbgs_in_nCtypes(DAMbgsAcrossNanalysis = dam_bg$DAMbgsAcrossNanalysis, figuresPath = figuresPath, produce_plots = produce_plots)
summarize_DAMbgs_multiplicity(DAMbgsAcrossNanalysis = dam_bg$DAMbgsAcrossNanalysis, intogen_drivers = intogen_drivers_s)

# Expression of hosting genes and essentiality validation
exp_stats <- plot_DAM_expression(allHits = allHits, figuresPath = figuresPath, produce_plots = produce_plots)
match_rate <- plot_essentiality_matching(allHits = allHits, figuresPath = figuresPath, produce_plots = produce_plots)

# Global hit summaries
plot_all_hits_summary(allHits = allHits, clc = clc, figuresPath = figuresPath, produce_plots = produce_plots)

# Signal vs cohort size correlations
percHitsPerCls <- plot_hits_correlations(allDAMs = allDAMs, totalTestedVariants = totalTestedVariants, incl_cl_annot = incl_cl_annot, clc = clc, figuresPath = figuresPath, produce_plots = produce_plots)

# Genomic instability (SNR) comparison
plot_SNR_genomic_groups(percHitsPerCls = percHitsPerCls, clc = clc, figuresPath = figuresPath, produce_plots = produce_plots)

# Unreported DAM bearing genes across cancer types statistics
plot_most_frequent_unreported_DAMbgs(allHits = allHits, intogen_drivers = intogen_drivers_s, figuresPath = figuresPath, produce_plots = produce_plots)


2237 out of 2284 unique DAMs ( 97.94221 %) are detected in a single ctype specific analysis
1051 out of 1383 DAMbgs ( 75.99422 %) are detected in a single ctype specific analysis
332  DAMbgs ( 24.00578 %) are detected in multiple ctypes
Of these 291 (87.6506%) are unreported DAMbgs
79.91 % of DAMs are in genes expressed in the cell line in which the DAM is observed
60.69 % of DAMs are in genes with a basal expression over the 50th percentile
28.12 % of DAMs are in genes with a basal expression over the 80th percentile
8.89 % of DAMs are in genes at the highest percentile of basal expression
Perfect overlap between DAM-hosting CCLs and gene-dependent CCLs observed in 46.39 % of hits

--- SNR Group Comparisons ---

Genomically quiet vs Noisy t.test

	Welch Two Sample t-test

data:  percHitsPerCls[genomicallyQuite] and percHitsPerCls[genomicallyNoisy]
t = 2.3598, df = 22.819, p-value = 0.02722
alternative hypothesis: true difference in means is not equal to 0
95 percent confidence interva

## 10. Reactome enrichment
Performing Reactome pathway enrichment analysis for DAM-bearing genes, comparing enrichment in all DAM-bearing genes versus known cancer-driver DAM-bearing genes. Testing with permutation-based simulations and hypergeometric tests whether pathway enrichment and driver co-occurrence exceed chance expectations.

In [ ]:
# OPTIONAL
# If downloading ensembl again
library(biomaRt)
# ensembl <- useMart("ensembl", dataset = "hsapiens_gene_ensembl")
ensembl <- useEnsembl(biomart = "genes", dataset = "hsapiens_gene_ensembl", version = 114)

In [ ]:
source('./R/10_reactome_enrichment.R')
load(file.path(resultPath, '_allHits.RData'))
load(file.path(resultPath, '_allDAMs.RData'))
load(file.path(resultPath, '_totalTestedVariants.RData'))
load(file.path(resultPath, '_allDAM_bearing_genes.RData'))

# Background gene list (from the whole set of tested variants)
background <- unique(totalTestedVariants$gene_symbol)

# Convert gene lists to Entrez IDs
gene_symbols <- unique(allDAM_bearing_genes$allDAM_bearing_genes)
#allDAM_bearing_entrez <- convert_symbols_to_entrez(symbols = gene_symbols, ensembl = ensembl)
# To use in case biomaRt is not available
allDAM_bearing_entrez <- convert_symbols_to_entrez2(symbols = gene_symbols)

#background_entrez <- convert_symbols_to_entrez(symbols = background, ensembl = ensembl)
# To use in case biomaRt is not available
background_entrez <- convert_symbols_to_entrez2(symbols = background)

# New (unreported) DAM-bearing genes
new_gene_symbols <- setdiff(unique(allDAM_bearing_genes$allDAM_bearing_genes), intogen_drivers_s)
#new_DAM_bearing_entrez <- convert_symbols_to_entrez(symbols = new_gene_symbols, ensembl = ensembl)
# To use in case biomaRt is not available
new_DAM_bearing_entrez <- convert_symbols_to_entrez2(symbols = new_gene_symbols)

# Convert IntOGen drivers to Entrez IDs
#IntOGen_Drivers_entrez <- convert_symbols_to_entrez(symbols = intogen_drivers_s, ensembl = ensembl)
# To use in case biomaRt is not available
IntOGen_Drivers_entrez <- convert_symbols_to_entrez2(symbols = intogen_drivers_s)

# Known DAM-bearing genes (intersection on Entrez)
known_DAM_bearing_entrez <- intersect(allDAM_bearing_entrez, IntOGen_Drivers_entrez)

# Run Reactome enrichment for all/new/known sets
all_DAM_enrichments <- as.data.frame(run_reactome_enrichment(gene_entrez = allDAM_bearing_entrez, universe_entrez = background_entrez, prefix = 'all', resultPath = resultPath,
                                               figuresPath = figuresPath, produce_plots = produce_plots))
new_DAM_enrichments <- as.data.frame(run_reactome_enrichment(gene_entrez = new_DAM_bearing_entrez, universe_entrez = background_entrez, prefix = 'new', resultPath = resultPath,
                                               figuresPath = figuresPath, produce_plots = produce_plots))
known_DAM_enrichments <- as.data.frame(run_reactome_enrichment(gene_entrez = known_DAM_bearing_entrez, universe_entrez = background_entrez, prefix = 'known', resultPath = resultPath,
                                                 figuresPath = figuresPath, produce_plots = produce_plots))

# Compare enrichment results
compenr <- compare_enrichments_and_write(all_enr = all_DAM_enrichments, known_enr = known_DAM_enrichments, resultPath = resultPath, figuresPath = figuresPath,
                                         intOGen_drivers = intogen_drivers_s, produce_plots = produce_plots)

# Prepare pathway sets - enriched only in the set with all the genes or conserved also in the set of known genes
Conserved_paths <- intersect(known_DAM_enrichments$Description, all_DAM_enrichments$Description)
OnlyInAllPaths <- setdiff(all_DAM_enrichments$Description, known_DAM_enrichments$Description)

# Prepare data for enrichment and co-occurrence analyses
# bgData <- prepare_background_data(background = background, intOGen_drivers = intogen_drivers_s, ensembl = ensembl)

# To use in case biomaRt is not available
bgData <- prepare_background_data2(background = background, intOGen_drivers = intogen_drivers_s)

# Permutation-based Reactome enrichment simulation
simRes <- run_random_enrichment_simulation(new_DAM_bearing_entrez = new_DAM_bearing_entrez, known_DAM_bearing_entrez = known_DAM_bearing_entrez, new_background_entrez = bgData$new_background_entrez,
                                 all_genes = bgData$all_genes, nsim = 1000, pathway_to_genes = bgData$pathway_to_genes, known_DAM_enrichments = known_DAM_enrichments, background_entrez = background_entrez,
                                 OnlyInAllPaths = OnlyInAllPaths, resultPath = resultPath)

cleng_ <- simRes$cleng_
eleng_ <- simRes$eleng_
presencePath <- simRes$presencePath

save(cleng_, file = file.path(resultPath, '_pws_enrich_all_vs_known_DAM_bearingG_enrichedPathways_overlap_random.RData'))
save(eleng_, file = file.path(resultPath, '_pws_enrich__all_vs_known_DAM_bearingG_enrichedPathways_NewOnly_random.RData'))
save(presencePath, file = file.path(resultPath, '_pws_enrich__all_vs_known_DAM_bearingG_enrichedPathways_NewOnly_random_pPath.RData'))

# Pathway empirical enrichment analysis
enrichmentEmpiricalRes <- run_pathway_enrichment_empirical_tests(cleng_ = cleng_, eleng_ = eleng_, Conserved_paths = Conserved_paths, newOnly = OnlyInAllPaths,
                                                                 new_DAM_bearing_entrez = new_DAM_bearing_entrez, presencePath = presencePath, figuresPath = figuresPath, produce_plots = produce_plots)

# Test co-occurrence of new DAM-bearing genes in pathways with known drivers
coOccRes <- run_driver_cooccurrence_analysis(pathway_to_genes = bgData$pathway_to_genes, IntOGen_Drivers_entrez = IntOGen_Drivers_entrez, new_DAM_bearing_entrez = new_DAM_bearing_entrez,
                                             all_genes = bgData$all_genes, new_gene_symbols = new_gene_symbols, figuresPath = figuresPath, nperm = 1000, produce_plots = produce_plots)

# Pathway coverage analysis
coverageRes <- run_pathway_coverage_analysis(known_DAM_enrichments = known_DAM_enrichments, all_DAM_enrichments = all_DAM_enrichments, Conserved_paths = Conserved_paths,
                                             newOnly = compenr$newOnly, known_DAM_bearing_entrez = known_DAM_bearing_entrez, new_DAM_bearing_entrez = new_DAM_bearing_entrez,
                                             reactome.db = reactome.db, figuresPath = figuresPath, produce_plots = produce_plots)


## 11. Cross-tissue DAM analysis
Further exploring DAMs across tissues: counting DAM-bearing genes across tissues after normalization by mutational burden, comparing hits with known cancer-driver gene lists, and summarizing recurrent driver/non-driver hits across cancer types.

In [ ]:
source("./R/11_crosstissue_DAMs.R")
load(file.path(resultPath, '_incl_cl_annot.RData'))
load(file.path(resultPath, '_allDAMs.RData'))

# Load per tissue DAM results
lr <- load_results(home = home, resultPath = resultPath)
tissues <- lr$tissues
results <- lr$results

# Compute tissue-level mutational burden
mb <- compute_mut_burden(tissues = tissues, CMP_annot = CMP_annot0, cl_variants = cl_variants)
mut_burden_anno <- mb$mut_burden_anno
mut_burden_new  <- mb$mut_burden_new

# Count hits per tissue and normalize by mutational burden
num_hits <- compute_hits_by_tissue(results = results, tissues = tissues, dep_threshold = dep_threshold, RR_th = RR_th, hypTest_p = hypergeom_p_th, empPval = empirical_p_th)
num_hits_norm <- num_hits / mut_burden_new

report_hits_by_tissue(num_hits_norm = num_hits_norm)

# Collect hit genes across all tissues
hits <- collect_hits(results = results, tissues = tissues, dep_threshold = dep_threshold, RR_th = RR_th, hypTest_p = hypergeom_p_th, empPval = empirical_p_th)
report_hit_summary(hits = hits, driver_genes = intogen_drivers_s)

# Plot hit recurrence and overlap with known drivers
plot_hit_frequency(hits = hits, driver_genes = intogen_drivers_s, figuresPath = figuresPath, produce_plots = produce_plots)
perc_int <- plot_driver_hits_venn(figuresPath = figuresPath, driver_genes = intogen_drivers_s, hits = hits, produce_plots = produce_plots)

# How many drivers per cancer type?
cancer_match <- read.csv(file.path(raw_dir, config$files$cancer_match_file_v3), header = TRUE, row.names = 1, sep = ";")

cancer_match_long_CMP  <- c()
cancer_match_long_into <- c()
for (i in 1:nrow(cancer_match)) {
  cancer_match_long_into <- c(cancer_match_long_into, unlist(strsplit(cancer_match[i, 1], " \\| ")))
  cancer_match_long_CMP  <- c(cancer_match_long_CMP, rep(rownames(cancer_match)[i], length(unlist(strsplit(cancer_match[i, 1], " \\| ")))))
}
cancer_match_long <- cbind(cancer_match_long_CMP, cancer_match_long_into)

num_drivers <- table(unlist(strsplit(unique(paste(intogen_drivers$SYMBOL, intogen_drivers$CANCER_TYPE)), " "))[seq(2, (2*nrow(intogen_drivers)), 2)])

# Compare hits against other benchmark driver-gene lists
benchmark <- read.xlsx(file.path(raw_dir, config$files$benchmark_file), sheet = 1)
plot_other_driver_venns_and_tests(benchmark = benchmark, hits = hits, cl_variants = cl_variants, figuresPath = figuresPath, produce_plots = produce_plots)

# Drivers across tissues: heatmap and barplots
drvRes <- compute_driver_summary_matrices(driver_genes = intogen_drivers_s, tissues = tissues, results = results, cancer_match_long_CMP = cancer_match_long_CMP,
                                          cancer_match_long_into = cancer_match_long_into, intogen_drivers = intogen_drivers, dep_threshold = dep_threshold,
                                          RR_th = RR_th, hypTest_p = hypergeom_p_th, empPval = empirical_p_th)

summary_drivers_bin <- drvRes$summary_drivers_bin
save(summary_drivers_bin, file = file.path(resultPath, "_summary_drivers_bin.RData"))

plot_driver_heatmap(figuresPath = figuresPath, summary_drivers_bin = drvRes$summary_drivers_bin, selhits2 = drvRes$selhits2, produce_plots = produce_plots)
plot_driver_barplot(figuresPath = figuresPath, summary_drivers = drvRes$summary_drivers, tissues = tissues, produce_plots = produce_plots)

# Non-driver hits across tissues
hits_nodriver <- setdiff(unique(hits), intogen_drivers_s)
save(hits_nodriver, file = file.path(resultPath, "_hits_nodriver.RData"))

summary_nodrivers <- compute_nondriver_summary_matrix(hits_nodriver = hits_nodriver, tissues = tissues, results = results,
                                                      dep_threshold = dep_threshold, RR_th = RR_th, hypTest_p = hypergeom_p_th, empPval = empirical_p_th)
plot_nondriver_ntissues(figuresPath = figuresPath, summary_nodrivers = summary_nodrivers, produce_plots = produce_plots)

# Stacked normalized hit counts per tissue
plot_hits_tissue_stack(figuresPath = figuresPath, summary_drivers = drvRes$summary_drivers, summary_nodrivers = summary_nodrivers, tissues = tissues, mut_burden_new = mut_burden_new,
                       num_hits_norm = num_hits_norm, produce_plots = produce_plots)

# Summary figure of DAMs across cell lines and cancer types
plot_DAM_summary(figuresPath = figuresPath, allDAMs = allDAMs, incl_cl_annot = incl_cl_annot, produce_plots = produce_plots)

the tissue with the highest percentage of DAM-bearing genes (considering mutational burden) is  Non-Small Cell Lung Carcinoma with 0.0359481 DAM-bearing genes
the tissue with the lowest percentage DAM-bearing genes (considering mutational burden) is  Burkitt's Lymphoma with 0.003149424 DAM-bearing genes
Number of unique DAM-bearing genes: 1383 
Number of DAM-bearing genes in at least two cancer types: 332 
of which not known to be drivers: 291 
Number of DAM-bearing genes known as driver: 123 
Number of DAM-bearing genes not known as driver: 1260 
Fraction of unique DAM-bearing genes annotated as known IntOGen cancer drivers: 0.06477093
Testing enrichment against external benchmark driver lists.2020Rule 8.004216e-18
Testing enrichment against external benchmark driver lists.benchmark 4.829853e-21
Testing enrichment against external benchmark driver lists.CGCpointMut 7.005881e-13
Testing enrichment against external benchmark driver lists.CGC 6.602002e-25
Testing enrichment against exter

agg_record_15afd7d9f4ff7 
                       2

## 12. STRING analysis
Testing whether non-driver hit genes are more strongly connected to known driver genes in the STRING protein-protein interaction network than expected by chance.

In [ ]:
source("./R/12_string_analysis.R")

load(file.path(resultPath,'_allHits.RData'))
load(file.path(resultPath,'_totaltestedVariants.RData'))

# Initialize the STRING database object
string_db <- STRINGdb$new(version = "12", species = 9606, score_threshold = 200, input_directory = "")

# Run the complete STRING interaction analysis:
# 1. compute the observed interaction score between hits and driver genes
# 2. generate random gene sets
# 3. compute an empirical p-value comparing observed vs random scores
res <- run_STRING_analysis(allHits = allHits, driver_genes = intogen_drivers_s, totalTestedVariants = totalTestedVariants, string_db = string_db, figuresPath = figuresPath, n_random = 1000, produce_plots = produce_plots)

## 13. Co-occurrence analysis
Identifying cell lines carrying DAMs in unreported DAM backgrounds and assessing co-occurring oncogenic addiction across cancer types.

In [ ]:
source("./R/13_cooccurrence.R")
load(file.path(resultPath, '_allDAMs.RData'))

tract <- read.table(file.path(raw_dir, config$files$tractability_file), sep = "\t", header = TRUE, row.names = NULL)
tractable_targets <- tract$id[which(tract$min_bucket==1)]
cancer_match <- read.csv(file.path(raw_dir, config$files$cancer_match_file_v3), header = TRUE, row.names = 1, sep = ";")

# Compute co-occurrence for all DAM-bearing cell lines
res_occ <- compute_DAM_oncogenic_cooccurrence(allDAMs = allDAMs, bdep = bdep, CMP_annot = CMP_annot0, cl_variants = cl_variants, driver_genes = intogen_drivers_s,
  intOGen_drivers = intogen_drivers, mapping = cancer_match, tractable_targets = tractable_targets)

# Save results
save(res_occ, file = file.path(resultPath,"_DAM_canonical_oncogenic_addition_co_occurrence.RData"))
write.table(res_occ, file = file.path(resultPath,"_DAM_canonical_oncogenic_addition_co_occurrence.tsv"), quote = FALSE, row.names = FALSE, sep = '\t')

# Summarize co-occurrence data and generate plots
coc_results <- summarize_cooccurrence(RES = res_occ, figuresPath = figuresPath, produce_plots = produce_plots)

67.06 % of CCLs with an unreported DAM lack DAMs in tissue-specific known GoF drivers
21.50 % of CCLs with an unreported DAM lack tissue-specific essential GoF drivers
Median % of CCLs lacking co-occurring DAMs in lineage-specific GoF drivers: 78.89%
Median % of CCLs lacking mutated and essential lineage-specific GoF drivers: 18.82%


## 14. DR validation
Validating significant DAM hits using drug-response data and annotating validated hits as SAMs and potential repurposable drug targets.


In [ ]:
set.seed(123)
source("./R/14_DR_validation.R")

load(file.path(resultPath, "_allHits.RData"))
DRplotsPath <- file.path(resultPath, config$paths$DRplots_dir)

cancer_match <- read.csv(file.path(raw_dir, config$files$cancer_match_file_v3), header = TRUE, row.names = 1, sep = ";")

# Build drug-response validation dataset
dr_results <- build_dr_validations(tissues = tissues, resultPath = resultPath, DRplotsPath = DRplotsPath, RR_th = RR_th,
                                   dep_threshold = dep_threshold, hypTest_p = hypergeom_p_th, empPval = empirical_p_th)
allDRvalidations <- dr_results$allDRvalidations
tissueDRvalidations <- dr_results$tissueDRvalidations

# Save tissue-specific files and global files
for (ctiss in names(tissueDRvalidations)) {
  rRES <- tissueDRvalidations[[ctiss]]
  save(rRES, file = file.path(DRplotsPath, paste0(ctiss, "_DR_validation.RData")))
  write.table(rRES, file = file.path(DRplotsPath, paste0(ctiss, "_DR_validation.tsv")), quote = FALSE, sep = "\t", row.names = FALSE)
}

save(allDRvalidations, file = file.path(resultPath, "_all_DR_validations.RData"))
write.table(allDRvalidations, file = file.path(resultPath, "_all_DR_validations.tsv"), quote = FALSE, sep = "\t", row.names = FALSE)

# Summarise validation results
stats <- summarise_dr_validations(allDRvalidations = allDRvalidations, allHits = allHits)

# SAM generation + driver annotation
allSAMs <- build_annotated_SAMs(allDRvalidations = allDRvalidations, intOGen_drivers = intogen_drivers, intOGen_drivers_s = intogen_drivers_s, ctypeMapping = cancer_match)

# Save results
save(allSAMs, file = file.path(resultPath, "_all_SAMs.RData"))
write.table(allSAMs, file = file.path(resultPath, "_all_SAMs.tsv"), quote = FALSE, sep = "\t", row.names = FALSE)

# Plotting genes most frequently involved in SAMs
if (produce_plots) {
  frequent_SAM_genes <- plot_frequent_SAM_genes(allSAMs = allSAMs, figuresPath = figuresPath)
}

Of the 1899 cancer-type-specific hits (DAMs or DAMs combinations with optimal RankRatio, fitness effect and pvalues), 74 involve a DAM-bearing gene that is druggable and targeted by a compound with available cancer-type matching drug-response data on GDSC.
Of these, 29 are validated.
Encompassing 63 individual cancer-type-specific drug validated DAMs (SAMs)
Encompassing 56 individual cancer-type-specific drug validated DAMs (SAMs) involving known cancer drivers


## 15. DAM position annotation
Adding genomic coordinates, allele strings, and Ensembl strand information to DAMs

In [ ]:
source("./R/15_pos_annot.R")
load(file.path(resultPath, '_allDAMs.RData'))

# Filter variants and match to DAMs
allDAMs_Positions <- filter_and_match_DAMs(cl_variants0 = cl_variants0, allDAMs = allDAMs)

# Save intermediate matched DAM positions
save(allDAMs_Positions, file = file.path(resultPath, "_allDAMs_PositionsV2.RData"))
write.table(allDAMs_Positions, file = file.path(resultPath, "_allDAMs_PositionsV2.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)

# Annotate genomic positions, alleles, and strand
allDAMs_Positions_annot <- annotate_DAM_positions(cl_vars = allDAMs_Positions)

# Order final output
allDAMs_Positions_annot <- allDAMs_Positions_annot[order(allDAMs_Positions_annot$gene_name),]
rownames(allDAMs_Positions_annot) <- NULL

# Save final annotated output
save(allDAMs_Positions_annot, file = file.path(resultPath, "_allDAMs_PositionsV3.RData"))
write.table(allDAMs_Positions_annot, file = file.path(resultPath, "_allDAMs_PositionsV3.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)

Ensembl site unresponsive, trying asia mirror



## 16. Add SIFT and PolyPhen scores
Querying Ensembl VEP to retrieve SIFT and PolyPhen predictions for DAMs, classifying each variant into functional impact categories, and transferring the resulting annotations to matching SAMs.


In [ ]:
source("./R/15_pos_annot.R")
source("./R/16_add_scores.R")

load(file.path(resultPath, '_allDAMs.RData'))
load(file.path(resultPath, '_all_SAMs.RData'))

# Filter variants and match to DAMs
allDAMs_Positions_v2 <- filter_and_match_DAMs_v2(cl_variants0 = cl_variants0, allDAMs = allDAMs)

# Reformat variant coordinates for Ensembl VEP
allDAMs_Positions_annot_v2 <- annotate_DAM_positions_v2(cl_vars = allDAMs_Positions_v2)

# Query Ensembl VEP and retrieve SIFT and PolyPhen predictions
annotated <- vep_sift_polyphen(df = allDAMs_Positions_annot_v2)

# Merge summarised VEP predictions back into the original DAM table
allDAMs_with_scores <- merge_DAMs_with_VEP(allDAMs = allDAMs, annotated_results = annotated$results)

# Classify functional impact
allDAMs_with_scores$impact_call <- mapply(classify_variant, allDAMs_with_scores$sift_min, allDAMs_with_scores$sift_pred_any, allDAMs_with_scores$polyphen_max, allDAMs_with_scores$polyphen_pred_any)

# Save DAM-level results
save(allDAMs_with_scores, file = file.path(resultPath, "_allDAMs_with_SIFT_PolyPhen_scores.RData"))
write.table(allDAMs_with_scores, file = file.path(resultPath, "_allDAMs_with_SIFT_PolyPhen_scores.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)

# Plot DAM summaries
plot_DAM_VEP_summary(allDAMs_with_scores = allDAMs_with_scores, figuresPath = figuresPath)
plot_DAM_VEP_summary(allDAMs_with_scores = allDAMs_with_scores, figuresPath = figuresPath, driver_genes = intogen_drivers_s, prefix = "DAM_unreported")

# Transfer DAM-based impact labels to SAMs and save
allSAMs_with_scores <- annotate_SAMs_with_VEP(allSAMs = allSAMs, allDAMs_with_scores = allDAMs_with_scores)

save(allSAMs_with_scores, file = file.path(resultPath, "_allSAMs_with_SIFT_PolyPhen_scores.RData"))
write.table(allSAMs_with_scores, file = file.path(resultPath, "_allSAMs_with_SIFT_PolyPhen_scores.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)

# Plot SAM summaries
plot_SAM_VEP_summary(allSAMs_with_scores = allSAMs_with_scores, figuresPath = figuresPath, produce_plots = produce_plots)
plot_SAM_VEP_summary(allSAMs_with_scores = allSAMs_with_scores, figuresPath = figuresPath, driver_genes = intogen_drivers_s, prefix = "SAM_unreported", produce_plots = produce_plots)

Ensembl site unresponsive, trying www mirror

Ensembl site unresponsive, trying asia mirror

Submitting 2284 variants in 46 batch(es) to Ensembl VEP REST…



  |                                                                      |   0%

Batch 1 | size=50 | attempt=1



  |==                                                                    |   2%

Batch 2 | size=50 | attempt=1



...

  |====================================================================  |  98%

Batch 46 | size=34 | attempt=1



  |======================================================================| 100%


Parsing and summarising transcript-level predictions…

Done. Annotated: 2284 | Failed: 0




DAM : 1086 DAMs have a high/moderate functional impact
DAM : 1594 DAMs have a high/moderate/possible functional impact
DAM_unreported : 895 DAMs have a high/moderate functional impact
DAM_unreported : 1302 DAMs have a high/moderate/possible functional impact


##

## 17. IntOGen and COSMIC patient analysis
Counting how often each DAM appears in intOGen and COSMIC, harmonizing tissue / cancer type labels, and creating ranked recurrence matrices for comparison.


In [6]:
mem.maxVSize(32000)

[1] 32000

In [ ]:
source("./R/17_intogen_cosmic_patients.R")

load(file.path(resultPath, "_allDAMs_PositionsV3.RData"))
load(file = file.path(resultPath, "_allDAMs.RData"))

# Data loading
cancer_match <- read.csv(file.path(raw_dir, config$files$cancer_match_file_v3), header = TRUE, row.names = 1, sep = ";")
loc_file <- read.delim(file.path(pathdata, config$files$loc_file))

COSMIC <- read_tsv(file.path(raw_dir, config$files$cosmic_mutant_file_new), show_col_types = FALSE)
COSMIC <- data.frame(COSMIC)
COSMICsamples <- read_tsv(file.path(raw_dir, config$files$cosmic_sample_file_new), show_col_types = FALSE)
COSMICsamples <- data.frame(COSMICsamples)
COSMICclassification <- read_tsv(file.path(raw_dir, config$files$cosmic_classification_file_new), show_col_types = FALSE)
COSMICclassification <- data.frame(COSMICclassification)

# Map each COSMIC mutation record to its primary tissue/site
COSMICprimarysite <- (COSMICclassification$PRIMARY_SITE[match(COSMIC$COSMIC_PHENOTYPE_ID, COSMICclassification$COSMIC_PHENOTYPE_ID)])

# Build the raw aggregated intOGen count table and assign variant labels
intogen_counts_aggr_raw <- create_intogen_counts_aggr(loc_file = loc_file, allDAMs_Positions = allDAMs_Positions_annot)

# Summarize recurrence across tissues and external datasets
res <- compute_summary_and_tissue_lists(allDAMs = allDAMs, intogen_counts_aggr = intogen_counts_aggr_raw, COSMIC = COSMIC, COSMICprimarysite = COSMICprimarysite, ntiss = ntiss)

summary_vars <- res$summary_vars
all_tiss_intogen <- res$all_tiss_intogen
all_tiss_cosmic <- res$all_tiss_cosmic

# Finalize the IntOGen count matrix for export/comparison
intogen_counts_aggr <- finalize_intogen_counts_aggr(intogen_counts_aggr = intogen_counts_aggr_raw)

# Convert COSMIC tissue-specific counts into an aggregated matrix
cosmic_counts_aggr <- create_cosmic_counts_aggr(all_tiss_cosmic = all_tiss_cosmic)

# Extend the cancer-type mapping table with unmapped categories
cancer_match_v3 <- create_cmp_mapping_v3(cmp_mapping = cancer_match, intogen_counts_aggr = intogen_counts_aggr, cosmic_counts_aggr = cosmic_counts_aggr)

# Aggregate IntOGen and COSMIC counts into comparable cancer-type groups
aggregated_counts_aggr_comp <- compute_aggregated_counts_aggr_comp(intogen_counts_aggr = intogen_counts_aggr, cosmic_counts_aggr = cosmic_counts_aggr, cmp_mapping = cancer_match_v3)

# Save results
save(summary_vars, file = file.path(resultPath, "cosmic_new/summary_byvar.RData"))
save(all_tiss_intogen, file = file.path(resultPath, "cosmic_new/all_tiss_intogen_byvar.RData"))
save(all_tiss_cosmic, file = file.path(resultPath, "cosmic_new/all_tiss_comsic_byvar.RData"))
save(intogen_counts_aggr, file = file.path(resultPath, "cosmic_new/_intogen_count_aggr.RData"))
save(cosmic_counts_aggr, file = file.path(resultPath, "cosmic_new/_cosmic_count_aggr.RData"))
save(aggregated_counts_aggr_comp, file = file.path(resultPath, "cosmic_new/_aggregated_count_aggr.RData"))

write.table(summary_vars, sep = "\t", quote = FALSE, row.names = FALSE, file = file.path(tablesPath, "cosmic_new/DAMs_occurrences_in_Patients.tsv"))
write.table(intogen_counts_aggr, sep = "\t", quote = FALSE, file = file.path(tablesPath, "cosmic_new/DAMs_occurrences_in_Patients_IntoGen_aggr.tsv"))
write.table(cosmic_counts_aggr, sep = "\t", quote = FALSE, file = file.path(tablesPath, "cosmic_new/DAMs_occurrences_in_Patients_Cosmic_aggr.tsv"))
write.table(aggregated_counts_aggr_comp, sep = "\t", quote = FALSE, file = file.path(tablesPath, "cosmic_new/DAMs_occurrences_in_Patients_all_aggr.tsv"))
write.table(cancer_match_v3, file = file.path(raw_dir, "intOGen ctype mapping_AS_v3_cosmic_new.csv"), sep = ";")

## 18. Summary of patient data
Combining functional impact predictions with patient recurrence data from IntOGen/COSMIC, marking whether DAMs are found in matching tumour types and known driver genes.

In [ ]:
source("./R/18_patient_summary.R")

load(file.path(resultPath, "_intogen_count_aggr.RData"))
load(file.path(resultPath, "_cosmic_count_aggr.RData"))
load(file.path(resultPath, "_aggregated_count_aggr.RData"))
load(file.path(resultPath, "_allDAMs_with_SIFT_PolyPhen_scores.RData"))

ctype_matching <- read.delim(file.path(raw_dir, config$files$cancer_match_file_v3), sep = ";", row.names = 1)
clc <- read.xlsx(file.path(raw_dir, config$files$cl_tissue_colors_file), sheet = 2, rowNames = TRUE)

# Split aggregated counts by driver status
sp <- split_aggregated_by_driver(aggregated_counts_aggr_comp = aggregated_counts_aggr_comp, intogen_drivers_s = intogen_drivers_s)
count_agg_known <- sp$count_agg_known
count_agg_unreported <- sp$count_agg_unreported

# Assign plotting colors to the cancer types represented in the known-driver matrix
COLS <- compute_cols_for_known(count_agg_known = count_agg_known, clc = clc)

# Add patient recurrence and tumour-type matching information to the DAM table
res <- build_funcImpact_and_clinical_rel(allDAMs_with_SIFT_Polyphen = allDAMs_with_scores, intogen_counts_aggr = intogen_counts_aggr,
                                         cosmic_counts_aggr = cosmic_counts_aggr, aggregated_counts_aggr_comp = aggregated_counts_aggr_comp,
                                         ctype_matching = ctype_matching, intogen_drivers_s = intogen_drivers_s)

allDAMs_with_funcImpact_and_clinical_rel <- res$allDAMs_with_funcImpact_and_clinical_rel

# Save the combined functional impact / clinical recurrence table
save(allDAMs_with_funcImpact_and_clinical_rel, file = file.path(resultPath, "_allDAMs_with_funcImpact_and_clinical_rel.RData"))
write.table(allDAMs_with_funcImpact_and_clinical_rel, file = file.path(tablesPath, "DAMs_with_functional_impact_and_clinical_rec.tsv"), quote = FALSE, row.names = FALSE, sep = "\t")

# Plot pie-chart summaries showing how many DAMs are observed in patient datasets
plot_and_report_patient_observation_pies(allDAMs_with_funcImpact_and_clinical_rel = allDAMs_with_funcImpact_and_clinical_rel,
                                         figuresPath = figuresPath, intogen_drivers_s = intogen_drivers_s, produce_plots = produce_plots)

# Print counts used in the Funnel / Figure 1D summary
cat_funnel_figure1D_stats(allDAMs_with_funcImpact_and_clinical_rel = allDAMs_with_funcImpact_and_clinical_rel, intogen_drivers_s = intogen_drivers_s)

# Summarize how often each unique DAM is observed in patients and plot
prev_out <- compute_uvar_prevalence_and_outputs(allDAMs_with_SIFT_Polyphen = allDAMs_with_scores, allDAMs_with_funcImpact_and_clinical_rel = allDAMs_with_funcImpact_and_clinical_rel,
  intogen_counts_aggr = intogen_counts_aggr, cosmic_counts_aggr = cosmic_counts_aggr, path_results = resultPath,
  figuresPath = figuresPath, intogen_drivers_s = intogen_drivers_s, produce_plots = produce_plots)

# Generate circular summary plot (Figure 4A)
load(file.path(resultPath, "_DF_for_circular_plot.RData"))
if (produce_plots) {
  plot_circular_dams(df = df, figuresPath = figuresPath)
}

of the 2376 unique DAMs, 1032 (43.43%) are observed in cancer patients (COSMIC or intOGen) 
341 (14.35%) are observed in both 
of the 2002 unique DAMs involving unreported driver genes, 792 (39.56%) are observed in cancer patients (COSMIC or intOGen) 
167 (8.34%) are observed in both 
n cancer type specific DAMs 2376 
of which in unreportd DAMbgs 2002 
with predicted functional impact 1594 
of which in unreported DAMbgs 1302 
with predicted functional impact and mutated patients 694 
of which in unreported DAMbgs 486 
with predicted functional impact and mutated patients (matching cancer type) 361 
of which in unreported DAMbgs 205 


## 19. DAM patient actionability and tractability analysis
Finding DAM-bearing COSMIC patients, checking existing CIViC actionability, and adding drug / tractability evidence for DAM genes.

In [ ]:
mem.maxVSize(32000)

[1] 32000

In [ ]:
source("./R/19_clinical_act.R")

load(file = file.path(resultPath, "_allDAMs.RData"))

# Load CIViC variants and mapping
civic_data <- load_civic_data(raw_dir = raw_dir, civic_file = config$files$civic_file, civic_mapping_file = config$files$civic_mapping_file)
civicdb_var_sel <- civic_data$civicdb_var_sel
civicdb_mapping <- civic_data$civicdb_mapping

# Load DAMs
dams_data <- load_dams(intogen_drivers = intogen_drivers, intogen_drivers_s = intogen_drivers_s, path_results = resultPath)
act_drivers <- dams_data$act_drivers
allDAMs <- dams_data$allDAMs

# Clean COSMIC
COSMIC <- read_tsv(file.path(raw_dir, config$files$cosmic_mutant_file), show_col_types = FALSE)
COSMIC <- data.frame(COSMIC)
COSMICclassification <- read_tsv(file.path(raw_dir, config$files$cosmic_classification_file), show_col_types = FALSE)
COSMICclassification <- data.frame(COSMICclassification)
COSMICprimarysite <- (COSMICclassification$PRIMARY_SITE[match(COSMIC$COSMIC_PHENOTYPE_ID, COSMICclassification$COSMIC_PHENOTYPE_ID)])
ctype_matching <- read.csv(file.path(raw_dir, config$files$cancer_match_file_v3), header = TRUE, sep = ";", row.names = 1)
cosmic_clean <- clean_cosmic_data(COSMIC = COSMIC, COSMICprimarysite = COSMICprimarysite, allDAMs = allDAMs, act_drivers = act_drivers,
                                  civicdb_var_sel = civicdb_var_sel, CMP_COSMIC_ctype_mapping = ctype_matching)

COSMIC <- cosmic_clean$COSMIC
COSMICprimarysite <- cosmic_clean$COSMICprimarysite
CMP_ctype_in_COSMIC_Cunt <- cosmic_clean$CMP_ctype_in_COSMIC_Cunt
COSMIC_tissue_Count <- cosmic_clean$COSMIC_tissue_Count

# Save RData
save(COSMIC_tissue_Count, file = file.path(robj_dir, "__COSMIC_tissue_Count.RData"))
save(CMP_ctype_in_COSMIC_Cunt, file = file.path(robj_dir, "__COSMIC_CMP_ctype_in_COSMIC_Cunt.RData"))
save(COSMIC, file = file.path(robj_dir, "__COSMIC_reduced.RData"))
save(COSMICprimarysite, file = file.path(robj_dir, "__COSMICprimarysite_reduced.RData"))

# Extract CIViC patients
patients_civic <- extract_civic_patients(civicdb_var_sel = civicdb_var_sel, civicdb_mapping = civicdb_mapping, COSMIC = COSMIC,
                                         COSMICprimarysite = COSMICprimarysite)
save(patients_civic, file = file.path(resultPath, "_patients_civic.RData"))

# Select DAM-bearing patients in COSMIC
DAMbearigPatient_in_COSMIC <- select_dam_patients(allDAMs = allDAMs, CMP_COSMIC_ctype_mapping = ctype_matching, COSMIC = COSMIC,
                                                  COSMICprimarysite = COSMICprimarysite)
save(DAMbearigPatient_in_COSMIC, file = file.path(robj_dir, "__COSMIC_DAMbearingPatients_in_COSMIC.RData"))

# Annotate DAM patients with CIViC actionability and driver status
DAMbearigPatient_in_COSMIC <- annotate_patients(DAMbearigPatient_in_COSMIC = DAMbearigPatient_in_COSMIC, patients_civic = patients_civic,
                                                drivers = intogen_drivers_s)

# Compute summary statistics
stats <- compute_patient_statistics(DAMbearigPatient_in_COSMIC = DAMbearigPatient_in_COSMIC)

# Plot pie chart about clinical actionability in patients
plot_actionable_pie(DAMbearigPatient_in_COSMIC = DAMbearigPatient_in_COSMIC, alreadyCurable = stats$alreadyCurable, figuresPath = figuresPath,
                    produce_plots = produce_plots)

# Export the patient-level DAM table with actionability and driver annotations
out_file <- file.path(tablesPath, "CIVIC_results.tsv")
DAMbearigPatient_in_COSMIC_export <- DAMbearigPatient_in_COSMIC
colnames(DAMbearigPatient_in_COSMIC_export) <- c('patient id','cancer type','variant','gene', 'has already a CIVIC variant','DAM in a known cancer driver')
write.table(DAMbearigPatient_in_COSMIC_export, quote=FALSE, sep='\t', row.names=FALSE, file = out_file)

# Plot cancer type distributions for patients lacking actionable mutations
clc <- readxl::read_excel(file.path(raw_dir, config$files$cl_tissue_colors_file), sheet = 2)
CL_colors_v <- clc$Color_hex_code
names(CL_colors_v) <- clc$`Cancer Type`

plot_cancer_type_distribution(DAMbearigPatient_in_COSMIC = DAMbearigPatient_in_COSMIC_export, CL_colors_v = CL_colors_v, CMP_ctype_in_COSMIC_Cunt = CMP_ctype_in_COSMIC_Cunt,
                              figuresPath = figuresPath, produce_plots = produce_plots)

# Plot off-context DAMs in known cancer-driver genes
if (produce_plots) {
  off_context_results <- plot_off_context_driver_DAMs(DAMbearigPatient_in_COSMIC = DAMbearigPatient_in_COSMIC, intogen_drivers = intogen_drivers,
  ctype_matching = ctype_matching, CMP_ctype_in_COSMIC_Cunt = CMP_ctype_in_COSMIC_Cunt, CL_colors_v = CL_colors_v, figuresPath = figuresPath)
}

# Run Open Targets actionability analysis
set.seed(123)
DAM_results <- compute_dam_actionability(raw_dir = raw_dir, tract_file = config$files$open_tract_file, resultPath = resultPath, figuresPath = figuresPath,
                                         allDAMs_file = "_allDAMs_with_funcImpact_and_clinical_rel.RData", totalTestedVariants_file = "_totalTestedVariants.RData",
                                         intOGen_drivers = intogen_drivers_s, produce_plots = produce_plots)
# Save full table
allDAMs_COMPLETE <- DAM_results$allDAMs
save(allDAMs_COMPLETE, file=file.path(resultPath, "_allDAMs_COMPLETE.RData"))
write.table(allDAMs_COMPLETE, file=file.path(tablesPath, "allDAMs_COMPLETE.tsv"), quote=FALSE, sep='\t', row.names=FALSE)